# AOI-PCB-SSD: Model Training

This notebook runs the full training pipeline for the custom SSD IC corner-point
detector described in the 2024 IEEE HORA paper. The same notebook trains either
architecture — set `ARCHITECTURE` in the setup cell:

- **`"custom"`** — the six-block custom feature extractor (Figure 3).
- **`"transfer"`** — the MobileNetV2 transfer-learning backbone (Figure 4).

The model is trained entirely on **synthetic data** composited from PCB templates.

### Prerequisites
1. Install the package: `pip install -e ".[dev]"`
2. Generate the dataset: `python scripts/generate_dataset.py`

### Contents
1. Setup
2. Data loading
3. Model architecture
4. Training
5. Save model
6. Results

## 1. Setup

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from aoi_pcb_ssd.config_loader import Config
from aoi_pcb_ssd.data.data_generator import DataGenerator
from aoi_pcb_ssd.encoding.input_encoder import SSDInputEncoder
from aoi_pcb_ssd.model.loss import AOILoss
from aoi_pcb_ssd.model.metrics import class_mAP, mae
from aoi_pcb_ssd.model.ssd_custom import build_custom_model
from aoi_pcb_ssd.model.ssd_transfer import build_transfer_model

# Architecture to train: "custom" (Figure 3) or "transfer" (Figure 4).
ARCHITECTURE = "custom"

config = Config("../config.json")
m = config.model
t = config.training

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")

## 2. Data Loading

`SSDInputEncoder` turns the per-image corner annotations into the fixed
`(64, 12)` SSD target tensor, and `DataGenerator` loads the cropped images,
applies augmentation, and encodes the labels. The data is then split 80/20 with
the same shuffled seed used in the paper so results are reproducible.

In [ ]:
encoder = SSDInputEncoder(
    img_height=m.img_height,
    img_width=m.img_width,
    n_classes=m.n_classes,
    predictor_sizes=m.predictor_sizes,
    normalize_coords=m.normalize_coords,
)

generator = DataGenerator(
    parent_dir=str(Path("..") / config.generator.train_data.crop_save_dir),
    encoder=encoder,
    augmentation=config.augmentation.enabled,
    probability=config.augmentation.probability,
)
X, y = generator.get_data()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=t.val_split, shuffle=True, random_state=t.random_seed
)
print(f"Train: {X_train.shape}  Val: {X_val.shape}")

## 3. Model Architecture

Both architectures share the three-branch detection head (class scores, corner
offsets, grid-cell anchor centres) and emit the same `(64, 12)` output. The
`custom` model uses the six-block CNN feature extractor; the `transfer` model
swaps in a MobileNetV2 backbone (ImageNet weights). Input preprocessing
(mean subtraction / standardisation) is built into the model graph.

In [ ]:
image_size = (m.img_height, m.img_width, m.img_channels)

if ARCHITECTURE == "custom":
    model = build_custom_model(
        image_size=image_size,
        n_classes=m.n_classes,
        l2_regularization=m.l2_regularization,
        normalize_coords=m.normalize_coords,
        subtract_mean=m.subtract_mean,
        divide_by_stddev=m.divide_by_stddev,
    )
elif ARCHITECTURE == "transfer":
    model = build_transfer_model(
        image_size=image_size,
        n_classes=m.n_classes,
        l2_regularization=m.l2_regularization,
        normalize_coords=m.normalize_coords,
    )
else:
    raise ValueError(f"Unknown ARCHITECTURE: {ARCHITECTURE!r}")

model.summary()

## 4. Training

The model is compiled with the Adam optimiser, the custom `AOILoss`
(hard-negative-mined classification + L1/L2 corner localisation, weighted by
α), and the `class_mAP` / `mae` metrics. Early stopping and learning-rate
reduction are driven entirely by `config.json`. A timestamped run directory
under `experiments/` receives a copy of the config and the per-epoch CSV log.

In [ ]:
run_dir = Path("../experiments") / f"notebook_{ARCHITECTURE}_{datetime.now():%Y%m%d_%H%M%S}"
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy("../config.json", run_dir / "config.json")
model_path = run_dir / "model.keras"

aoi_loss = AOILoss(**config.get_init_kwargs("training.loss"))
model.compile(
    optimizer=Adam(**config.get_init_kwargs("training.optimizer")),
    loss=aoi_loss.compute_loss,
    metrics=[class_mAP, mae],
)

training_callbacks = [
    EarlyStopping(**config.get_init_kwargs("training.early_stopping")),
    ReduceLROnPlateau(**config.get_init_kwargs("training.lr_schedule")),
    CSVLogger(str(run_dir / f"training_{model_path.stem}.csv")),
]

history = model.fit(
    X_train,
    y_train,
    batch_size=t.batch_size,
    epochs=t.epochs,
    validation_data=(X_val, y_val),
    callbacks=training_callbacks,
    verbose=2,
)

## 5. Save Model

In [ ]:
model.save(str(model_path))
print(f"Model saved to: {model_path}")

## 6. Results

Training and validation curves for the total loss and the `class_mAP` metric.
Early stopping restores the best weights, so the final model corresponds to the
epoch with the lowest validation loss.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["loss"], label="Train", color="steelblue")
ax1.plot(history.history["val_loss"], label="Validation", color="tomato")
ax1.set_title("Loss per Epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True)

# Keras 3 derives the history key from the metric function name via
# camelCase -> snake_case conversion: class_mAP -> "class_m_ap".
ax2.plot(history.history["class_m_ap"], label="Train", color="steelblue")
ax2.plot(history.history["val_class_m_ap"], label="Validation", color="tomato")
ax2.set_title("class_mAP per Epoch")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("class_mAP")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

n_epochs = len(history.history["loss"])
best_val_loss = min(history.history["val_loss"])
best_epoch = history.history["val_loss"].index(best_val_loss) + 1
print(f"Trained for {n_epochs} epochs.")
print(f"Best val_loss: {best_val_loss:.6f} at epoch {best_epoch}")